# Exploratory Data Analysis — FD002

Structured exploration of the NASA C-MAPSS turbofan engine degradation dataset (FD002 subset). Analysis logic is in `src/turbofan/eda/`; this notebook handles visualization.

## 1. Setup & Data Loading

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio

from turbofan.config.schema import load_config
from turbofan.data.loader import load_raw_train, load_raw_test, load_rul_labels
from turbofan.eda import quality, sensors, degradation

pio.templates.default = "plotly_white"
plt.rcParams["figure.figsize"] = (12, 6)

if Path.cwd().name == "notebooks":
    %cd ..

cfg = load_config(Path("configs/default.yaml"))
cfg = cfg.model_copy(
    update={
        "data": cfg.data.model_copy(update={"fd_subset": "FD002"})
    }
)

print(cfg.data.fd_subset)

In [ ]:
train_df = load_raw_train(cfg.data)
test_df = load_raw_test(cfg.data)
test_rul = load_rul_labels(cfg.data)

print(f"Train: {train_df.shape[0]:,} rows, {train_df['engine_id'].nunique()} engines")
print(f"Test:  {test_df.shape[0]:,} rows, {test_df['engine_id'].nunique()} engines")
print(f"RUL labels: {len(test_rul)} engines")
train_df.head()

## 2. Data Quality

Check for missing values, constant sensors, and data types.

In [ ]:
missing = quality.find_missing_values(train_df)
print("Missing values per column:")
print(missing[missing > 0] if missing.any() else "None \u2014 dataset is complete.")

In [ ]:
tol = cfg.features.sensor_std_threshold
low_var = quality.find_low_variance_sensors(train_df, tol=tol)
print(f"Sensors with std < {tol:.0e}: ({len(low_var)}): {low_var}")
print("These carry no information and should be dropped before modeling.")

In [ ]:
dtype_summary = quality.summarize_dtypes(train_df)
dtype_summary

## 3. Operational Settings

Distribution and unique combinations of the three operational setting columns.

In [ ]:
op_cols = ["op_1", "op_2", "op_3"]

print("Unique values per operational setting:")
for col in op_cols:
    exact_unique = train_df[col].nunique()
    rounded_unique = train_df[col].round(0).nunique()

    print(
        f"  {col}: {exact_unique} exact unique values "
        f"({rounded_unique} after rounding to 0 decimals)"
    )

In [ ]:
op_rounded = train_df[op_cols].round(0)

op_combos = (
    op_rounded
    .groupby(op_cols)
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

print(f"Approximate operating condition combinations: {len(op_combos)}")
op_combos

## 4. Sensor Distributions

Summary statistics and visual distributions for all 21 sensors.

In [ ]:
stats = sensors.compute_sensor_stats(train_df)
stats.round(3)

In [ ]:
sensor_cols = [c for c in train_df.columns if c.startswith("s_")]
non_low_var = [c for c in sensor_cols if c not in low_var]

fig, axes = plt.subplots(
    nrows=len(non_low_var) // 3 + 1,
    ncols=3,
    figsize=(15, 4 * (len(non_low_var) // 3 + 1)),
)
axes = axes.flatten()
for i, col in enumerate(non_low_var):
    train_df[col].hist(bins=50, ax=axes[i], edgecolor="black", alpha=0.7)
    axes[i].set_title(col)
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)
plt.tight_layout()
plt.suptitle("Sensor Distributions (informative sensors)", y=1.02)
plt.show()

## 5. Correlation Analysis

Sensor-sensor and sensor-RUL correlations to identify informative features.

In [ ]:
corr = sensors.compute_correlation_matrix(train_df, non_low_var)

fig, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(non_low_var)))
ax.set_yticks(range(len(non_low_var)))
ax.set_xticklabels(non_low_var, rotation=45, ha="right")
ax.set_yticklabels(non_low_var)
plt.colorbar(im, ax=ax, label="Pearson r")
ax.set_title("Sensor-Sensor Correlation Matrix")
plt.tight_layout()
plt.show()

In [ ]:
train_with_rul = degradation.compute_rul_curves(train_df, max_rul=125)
rul = train_with_rul["rul"]

informative = degradation.select_informative_sensors(
    train_df, rul, threshold=0.1
)
print(f"Informative sensors (|corr with RUL| > 0.1): {informative}")

rul_corr = train_df[non_low_var].corrwith(rul).sort_values()
fig = px.bar(
    x=rul_corr.values,
    y=rul_corr.index,
    orientation="h",
    labels={"x": "Pearson r with RUL", "y": "Sensor"},
    title="Sensor-RUL Correlation",
)
fig.show()

## 6. Degradation Trajectories

Visual inspection of how sensor readings evolve over an engine's lifetime.

In [ ]:
sample_engines = sorted(train_df["engine_id"].unique())[:5]
sample_df = train_df[train_df["engine_id"].isin(sample_engines)]

top_sensors = (
    rul_corr.abs()
    .sort_values(ascending=False)
    .head(4)
    .index
    .tolist()
)

fig, axes = plt.subplots(len(top_sensors), 1, figsize=(14, 4 * len(top_sensors)))
if len(top_sensors) == 1:
    axes = [axes]
for ax, sensor in zip(axes, top_sensors):
    for eid in sample_engines:
        engine_data = sample_df[sample_df["engine_id"] == eid]
        ax.plot(engine_data["cycle"], engine_data[sensor], alpha=0.7, label=f"Engine {eid}")
    ax.set_ylabel(sensor)
    ax.legend(loc="upper left", fontsize=8)
axes[-1].set_xlabel("Cycle")
plt.suptitle("Raw Sensor Degradation (sample engines)", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
smoothed = degradation.compute_sensor_trends(
    sample_df, top_sensors, window=10
)

fig, axes = plt.subplots(len(top_sensors), 1, figsize=(14, 4 * len(top_sensors)))
if len(top_sensors) == 1:
    axes = [axes]
for ax, sensor in zip(axes, top_sensors):
    for eid in sample_engines:
        engine_data = smoothed[smoothed["engine_id"] == eid]
        ax.plot(engine_data["cycle"], engine_data[sensor], alpha=0.7, label=f"Engine {eid}")
    ax.set_ylabel(f"{sensor} (smoothed)")
    ax.legend(loc="upper left", fontsize=8)
axes[-1].set_xlabel("Cycle")
plt.suptitle("Smoothed Sensor Trends (window=10)", y=1.01)
plt.tight_layout()
plt.show()

## 7. Operating-Mode Normalization Check

Fit `OperatingModeNormalizer` with `n_modes=6` on the training data, verify the discovered cluster centres match the known operating combos, and confirm that within each mode the normalized sensors have mean ≈ 0 and std ≈ 1.

In [ ]:
from turbofan.preprocessing.normalization import OperatingModeNormalizer

normalizer = OperatingModeNormalizer(
    feature_cols=non_low_var,
    op_cols=op_cols,
    n_modes=6,
    random_state=cfg.data.random_seed,
)
normalizer.fit(train_df)

centers_df = pd.DataFrame(normalizer.mode_centers_, columns=op_cols).round(2)
centers_df.index.name = "mode"
print("Discovered mode centres:")
print(centers_df)
print()
print("Known operating combos (from EDA):")
print(op_combos.drop(columns='count').to_string(index=False))

In [ ]:
train_norm = normalizer.transform(train_df)
mode_labels = normalizer._assign_modes(train_df)

check_sensors = top_sensors
fig, axes = plt.subplots(len(check_sensors), 2, figsize=(14, 4 * len(check_sensors)))

for row, sensor in enumerate(check_sensors):
    for m in range(6):
        mask = mode_labels == m
        axes[row, 0].hist(train_df.loc[mask, sensor], bins=40, alpha=0.5, label=f"mode {m}")
        axes[row, 1].hist(train_norm.loc[mask, sensor], bins=40, alpha=0.5, label=f"mode {m}")
    for col_idx, title_suffix in enumerate(["raw", "normalized"]):
        axes[row, col_idx].set_title(f"{sensor} — {title_suffix}")
        axes[row, col_idx].legend(fontsize=7)

plt.suptitle("Per-mode sensor distributions before and after normalization", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
train_norm_labeled = train_norm.copy()
train_norm_labeled["mode"] = mode_labels

check = top_sensors[:4]
print("Post-normalization per-mode mean (expect ≈ 0.0):")
print(train_norm_labeled.groupby("mode")[check].mean().round(3).to_string())
print()
print("Post-normalization per-mode std (expect ≈ 1.0):")
print(train_norm_labeled.groupby("mode")[check].std(ddof=0).round(3).to_string())

### 7.1 New Zero-Variance Sensors After Normalization

Some sensors may pass the global low-variance filter but become constant within every mode after per-mode normalization — their variance is driven entirely by operating-condition offsets, not degradation. Identify them here so they can be dropped alongside the pre-normalization low-variance set.

In [ ]:
post_std = train_norm[non_low_var].std()

new_zero_std = sorted(post_std[post_std < 1e-6].index.tolist())
low_signal = sorted(post_std[(post_std >= 1e-6) & (post_std < 0.5)].index.tolist())

print(f"Sensors with zero std after normalization (constant within every mode): {new_zero_std}")
print(f"Sensors with low but non-zero std after normalization (<0.5):          {low_signal}")
print()
print("These were not caught by the pre-normalization low-variance filter because")
print("their global variance is explained entirely by mode-to-mode offsets.")

### 7.2 Sensor-RUL Correlation — Raw vs Normalized

Re-compute Pearson correlations with RUL on the normalized data and compare against the raw results from section 5.

In [ ]:
norm_rul_corr = train_norm[non_low_var].corrwith(rul).sort_values()

comparison = pd.DataFrame({"raw": rul_corr, "normalized": norm_rul_corr}).sort_values("raw")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, col, title in zip(axes, ["raw", "normalized"], ["Raw", "Per-mode normalized"]):
    ax.barh(comparison.index, comparison[col])
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlabel("Pearson r with RUL")
    ax.set_title(f"Sensor-RUL Correlation — {title}")
plt.tight_layout()
plt.show()

delta = (norm_rul_corr.abs() - rul_corr.abs()).sort_values(ascending=False)
print("Absolute correlation change after normalization (+ = stronger):")
print(delta.round(3).to_string())

### 7.3 Degradation Trajectories — Normalized

Same sample engines as section 6; per-mode normalization should remove operating-condition jumps and expose the underlying degradation signal.

In [ ]:
sample_norm = train_norm[train_df["engine_id"].isin(sample_engines)].copy()

fig, axes = plt.subplots(len(top_sensors), 1, figsize=(14, 4 * len(top_sensors)))
if len(top_sensors) == 1:
    axes = [axes]
for ax, sensor in zip(axes, top_sensors):
    for eid in sample_engines:
        engine_data = sample_norm[sample_norm["engine_id"] == eid]
        ax.plot(engine_data["cycle"], engine_data[sensor], alpha=0.7, label=f"Engine {eid}")
    ax.set_ylabel(f"{sensor} (normalized)")
    ax.legend(loc="upper left", fontsize=8)
axes[-1].set_xlabel("Cycle")
plt.suptitle("Normalized Sensor Degradation Trajectories (sample engines)", y=1.01)
plt.tight_layout()
plt.show()

## 8. Summary & Key Findings

**Data quality:**
- No missing values in FD002 training data
- Operational settings collapse into 6 meaningful operating-condition combinations after rounding
- Sensor distributions are multimodal due to multiple operating regimes
- `s_16` is low-variance globally and should be dropped before modeling

**Operating-mode normalization:**
- `OperatingModeNormalizer` with `n_modes=6` recovered all 6 known operating points via KMeans
- 4 additional sensors become zero-variance after per-mode normalization — `s_1`, `s_5`, `s_18`, `s_19` — their global variance is driven entirely by mode-to-mode offsets, not degradation signal; they should be dropped alongside the pre-normalization low-variance set
- `s_10` (post-norm std 0.74, corr 0.026 with RUL) is effectively uninformative after normalization

**Sensor-RUL correlation after normalization:**
- Per-mode normalization produced large gains across all informative sensors (most gained > 0.5 in absolute Pearson r)
- Strongest negative correlators: `s_11` (−0.77), `s_4` (−0.74), `s_15` (−0.72), `s_17` (−0.65), `s_2` (−0.63)
- Strongest positive correlators: `s_12` (0.54), `s_7` (0.51), `s_20` (0.48), `s_21` (0.48)
- Uncorrelated post-normalization (< |0.05|): `s_1`, `s_5`, `s_10`, `s_18`, `s_19` — safe to drop

**Degradation patterns:**
- Per-mode normalized trajectories remove operating-condition jumps and expose cleaner degradation trends
- Normalization is a prerequisite for meaningful feature engineering on FD002

**Next steps:**
- Drop `s_16` (pre-norm low-variance) and `s_1`, `s_5`, `s_18`, `s_19` (post-norm zero-variance)
- Engineer rolling statistics and trend features from the remaining informative sensors on normalized data
- Feed normalized features into sequence models (GRU) for RUL prediction
